In [7]:
import pandas as pd
import numpy as np
import faiss
import pickle
from fastapi import FastAPI, HTTPException
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
DATA_FILE = 'books_data.csv'
INDEX_FILE = 'book_vectors.index'
SVD_MODEL_FILE = 'svd_model.pkl'

app = FastAPI()

# --- 1. GLOBAL LOAD (Run once when server starts) ---
print("⏳ Server starting... loading data and models...")

# Load Metadata
try:
    df_books = pd.read_csv(DATA_FILE)
    print(f"✅ Books Data Loaded: {len(df_books)} records")
except Exception as e:
    print(f"❌ Error loading CSV: {e}")

# Load FAISS Index (Search)
try:
    index = faiss.read_index(INDEX_FILE)
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print("✅ FAISS Index & Embedding Model Loaded")
except Exception as e:
    print(f"❌ Error loading Search components: {e}")

# Load SVD Model (Recommendation)
try:
    with open(SVD_MODEL_FILE, 'rb') as f:
        svd_model = pickle.load(f)
    print("✅ SVD Model Loaded")
except Exception as e:
    print(f"⚠️ SVD Model not found ({e}). /recommend endpoint will fail.")
    svd_model = None


# --- 2. API ENDPOINTS ---

@app.get("/")
def home():
    return {"message": "Book Recommendation API is Live 🚀"}

@app.get("/search")
def search_books(query: str, k: int = 5):
    """
    Endpoints that uses your FAISS work.
    Input: "scary clown"
    Output: JSON list of top 5 books.
    """
    if not query:
        return {"error": "Query cannot be empty"}
    
    # 1. Vectorize query
    vector = embedding_model.encode([query])
    
    # 2. Search Index
    # Note: FAISS expects float32
    distances, indices = index.search(np.array(vector).astype('float32'), k)
    
    results = []
    for i in range(k):
        idx = int(indices[0][i])
        # Map FAISS index back to DataFrame
        if idx < len(df_books):
            book = df_books.iloc[idx]
            results.append({
                "book_id": int(book.get('book_id', idx)), # Handle missing ID
                "title": str(book['title']),
                "match_score": float(distances[0][i])
            })
    
    return {"query": query, "results": results}

@app.get("/recommend/{user_id}")
def recommend(user_id: int):
    """
    Endpoint that uses your SVD/Collaborative Filtering work.
    Input: User ID (e.g., 123)
    Output: Top 5 predictions for that user.
    """
    if svd_model is None:
        return {"error": "Model not loaded"}
    
    # In a real app, we filter out books the user has already read.
    # For speed, we just score a sample of popular books (e.g., first 1000 IDs).
    candidate_books = df_books['book_id'].unique()[:1000]
    
    predictions = []
    for book_id in candidate_books:
        # Predict rating
        pred = svd_model.predict(uid=user_id, iid=book_id)
        predictions.append((book_id, pred.est))
    
    # Sort by highest predicted rating
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    # Format Top 5
    top_5 = predictions[:5]
    results = []
    for book_id, score in top_5:
        # Find title in dataframe
        book_info = df_books[df_books['book_id'] == book_id]
        if not book_info.empty:
            title = book_info.iloc[0]['title']
            results.append({
                "book_id": int(book_id),
                "title": title,
                "predicted_rating": round(score, 2)
            })
            
    return {"user_id": user_id, "recommendations": results}

⏳ Server starting... loading data and models...
✅ Books Data Loaded: 10000 records
✅ FAISS Index & Embedding Model Loaded
✅ SVD Model Loaded


In [4]:
pip install fastapi "uvicorn[standard]"

  Using cached starlette-0.50.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached httptools-0.7.1-cp312-cp312-win_amd64.whl.metadata (3.6 kB)
  Using cached watchfiles-1.1.1-cp312-cp312-win_amd64.whl.metadata (5.0 kB)
  Using cached websockets-16.0-cp312-cp312-win_amd64.whl.metadata (7.0 kB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
Using cached httptools-0.7.1-cp312-cp312-win_amd64.whl (86 kB)
Using cached starlette-0.50.0-py3-none-any.whl (74 kB)
Using cached watchfiles-1.1.1-cp312-cp312-win_amd64.whl (288 kB)
Using cached websockets-16.0-cp312-cp312-win_amd64.whl (178 kB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
uvicorn main:app --reload

SyntaxError: invalid syntax (3807844594.py, line 1)